In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

HGNC_PD = pd.DataFrame({
    "ensembl_gene_id": ["ENSG00000012048", "ENSG00000141510", "ENSG00000139618"],
    "entrez_id": ["672", "7157", "675"],
    "hgnc_id": ["HGNC:1100", "HGNC:11998", "HGNC:1101"],
    "symbol": ["BRCA1", "TP53", "BRCA2"],
    "refseq_accession": ["NM_007294", "NM_000546", "NM_000059"],
})
HGNC_PL = pl.from_pandas(HGNC_PD)

# --- pheval_gene_identifier_map ---
FIX_PHEVAL_GENE_IDENTIFIER_MAP_READ_HGNC_DATA_PD = lambda: HGNC_PD.copy()
FIX_PHEVAL_GENE_IDENTIFIER_MAP_READ_HGNC_DATA_PL = lambda: HGNC_PL.clone()

# --- pheval_gene_updater_find_identifier ---
FIX_PHEVAL_GENE_UPDATER_FIND_IDENTIFIER_GENE_SYMBOL = "BRCA1"

# --- pheval_gene_updater_find_symbol ---
FIX_PHEVAL_GENE_UPDATER_FIND_SYMBOL_QUERY_GENE_IDENTIFIER = "test_query"

# --- pheval_parse_hgnc_data ---

# Mock __file__ and create minimal hgnc TSV for pheval wrappers.
import tempfile as _tmpfile, os as _os
_hgnc_root = _tmpfile.mkdtemp()
_utils_dir = _os.path.join(_hgnc_root, "utils"); _os.makedirs(_utils_dir, exist_ok=True)
_res_dir   = _os.path.join(_hgnc_root, "resources"); _os.makedirs(_res_dir, exist_ok=True)
HGNC_PARSE_PD = pd.DataFrame({
    "hgnc_id": ["HGNC:1", "HGNC:2"],
    "symbol": ["A1BG", "BRCA1"],
    "entrez_id": ["1", "672"],
    "ensembl_gene_id": ["ENSG0", "ENSG00000012048"],
    "status": ["Approved", "Approved"],
    "refseq_accession": ["NM_130786", "NM_007294"],
})
HGNC_PARSE_PD.to_csv(_os.path.join(_res_dir, "hgnc_complete_set.txt"), sep="	", index=False)
__file__ = _os.path.join(_utils_dir, "phenopacket_utils.py")

print("✅ Fixtures loaded")
self = SimpleNamespace(
    hgnc_data={
        "BRCA1": {"entrez_id": "672", "previous_symbol": ["RNF53"]},
        "TP53": {"entrez_id": "7157", "previous_symbol": ["P53"]},
    },
    gene_identifier="entrez_id",
    identifier_map={"test_query": "BRCA1", "ncbigene:7157": "TP53"},
)


✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_pheval_gene_identifier_map(read_hgnc_data):
    hgnc_df = read_hgnc_data()
    identifier_map = {}
    for _index, row in hgnc_df.iterrows():
        identifier_map[row["ensembl_gene_id"]] = row["symbol"]
        identifier_map[row["hgnc_id"]] = row["symbol"]
        identifier_map[row["entrez_id"]] = row["symbol"]
        identifier_map[row["refseq_accession"]] = row["symbol"]
    return identifier_map
    return identifier_map

def before_pheval_gene_updater_find_identifier(gene_symbol):
    if gene_symbol in self.hgnc_data.keys():
        return self.hgnc_data[gene_symbol][self.gene_identifier]
    else:
        for _symbol, data in self.hgnc_data.items():
            for prev_symbol in data["previous_symbol"]:
                if prev_symbol == gene_symbol:
                    return data[self.gene_identifier]
    return None

def before_pheval_gene_updater_find_symbol(query_gene_identifier):
    return self.identifier_map[query_gene_identifier]
    return None

def before_pheval_parse_hgnc_data():
    return pd.read_csv(
        os.path.dirname(__file__).replace("utils", "resources/hgnc_complete_set.txt"),
        delimiter="\t",
        dtype=str,
    )
    return None

In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_pheval_gene_identifier_map(read_hgnc_data):

    hgnc_df = read_hgnc_data()
    identifier_map = {}
    for row in hgnc_df.iter_rows(named=True):
        identifier_map[row["ensembl_gene_id"]] = row["symbol"]
        identifier_map[row["hgnc_id"]] = row["symbol"]
        identifier_map[row["entrez_id"]] = row["symbol"]
        identifier_map[row["refseq_accession"]] = row["symbol"]
    return identifier_map

def gen_pheval_gene_updater_find_identifier(gene_symbol):
    if gene_symbol in self.hgnc_data.keys():
        return self.hgnc_data[gene_symbol][self.gene_identifier]
    else:
        for _symbol, data in self.hgnc_data.items():
            for prev_symbol in data["previous_symbol"]:
                if prev_symbol == gene_symbol:
                    return data[self.gene_identifier]
    return None

def gen_pheval_gene_updater_find_symbol(query_gene_identifier):
    return self.identifier_map[query_gene_identifier]

def gen_pheval_parse_hgnc_data():
    import os

    return pl.read_csv(
        os.path.dirname(__file__).replace("utils", "resources/hgnc_complete_set.txt"),
        separator="\t",
    ).select(pl.all().cast(pl.Utf8))

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:
# === Tests: pheval_parse_hgnc_data ===

# L1 smoke – generated
try:
    _r = gen_pheval_parse_hgnc_data()
    print("✅ L1 smoke gen_pheval_parse_hgnc_data: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_pheval_parse_hgnc_data: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_pheval_parse_hgnc_data()
    print("✅ L1 smoke before_pheval_parse_hgnc_data: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_pheval_parse_hgnc_data: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_pheval_parse_hgnc_data()
    _rg = gen_pheval_parse_hgnc_data()
    compare(_rb, _rg, "pheval_parse_hgnc_data")
except Exception as _e:
    print(f"❌ L2 equivalence pheval_parse_hgnc_data: setup error — {type(_e).__name__}: {_e}")

# L3 — extra TSV row and string-like numeric ids are preserved as strings.
try:
    old_text = Path(_os.path.join(_res_dir, "hgnc_complete_set.txt")).read_text()
    pd.concat([HGNC_PARSE_PD, pd.DataFrame({
        "hgnc_id": ["HGNC:3"], "symbol": ["ZZZ"], "entrez_id": ["00123"],
        "ensembl_gene_id": ["ENSGZ"], "status": ["Approved"], "refseq_accession": ["NM_Z"],
    })], ignore_index=True).to_csv(_os.path.join(_res_dir, "hgnc_complete_set.txt"), sep="	", index=False)
    _rb = before_pheval_parse_hgnc_data()
    _rg = gen_pheval_parse_hgnc_data()
    assert "00123" in _rg.get_column("entrez_id").to_list()
    compare(_rb, _rg, "L3 pheval_parse_hgnc_data string ids")
except Exception as _e:
    print(f"❌ L3 pheval_parse_hgnc_data string ids: {type(_e).__name__}: {_e}")
finally:
    Path(_os.path.join(_res_dir, "hgnc_complete_set.txt")).write_text(old_text)

# L3 — missing resource file should raise on both sides.
try:
    path = Path(_os.path.join(_res_dir, "hgnc_complete_set.txt"))
    backup = path.read_text()
    path.unlink()
    before_err = gen_err = None
    try:
        before_pheval_parse_hgnc_data()
    except Exception as e:
        before_err = type(e)
    try:
        gen_pheval_parse_hgnc_data()
    except Exception as e:
        gen_err = type(e)
    assert before_err is FileNotFoundError and gen_err is FileNotFoundError
    print("✅ L3 pheval_parse_hgnc_data missing file: both sides raise FileNotFoundError")
except Exception as _e:
    print(f"❌ L3 pheval_parse_hgnc_data missing file: {type(_e).__name__}: {_e}")
finally:
    path.write_text(backup)


✅ L1 smoke gen_pheval_parse_hgnc_data: OK, type= DataFrame
✅ L1 smoke before_pheval_parse_hgnc_data: OK
✅ L2 equivalence pheval_parse_hgnc_data: MATCH
❌ L3 pheval_parse_hgnc_data string ids: AssertionError: 
✅ L3 pheval_parse_hgnc_data missing file: both sides raise FileNotFoundError
